# 10-K Sentiment Analysis with FinBERT

**Goal:** measure the *tone* of a company's annual reports and see how it changes over time.

**Steps**
1. Download the company's 10-K filings from SEC EDGAR
2. Split each filing into its sections (Business, Risk Factors, ...)
3. Score every section with **FinBERT** (1 = negative, 3 = neutral, 5 = positive)
4. Visualise the results
5. Bonus: find the people mentioned and the latest news

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 1. Download the 10-K filings

We use the open-source [edgar-crawler](https://github.com/nlpaueb/edgar-crawler) tool (wrapped in `sec_download.py`).
It downloads each year's 10-K from SEC EDGAR and splits it into sections (`item_1`, `item_1A`, ...).

In [2]:
from sec_download import download_filings, load_filings

TICKER = "GOOG"
START_YEAR = 2019

download_filings(TICKER, START_YEAR)

## 2. Load the filings

Each filing is saved as a JSON file: one key per section.

In [3]:
filings = load_filings()
company = filings[0]["company"]
print(f"Loaded {len(filings)} filings for {company}:", [f["year"] for f in filings])

Loaded 8 filings for Alphabet Inc.: ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [4]:
# What does one filing look like?
print(list(filings[0].keys()))
print(filings[0]["item_2"][:500])

['cik', 'company', 'filing_type', 'filing_date', 'period_of_report', 'sic', 'state_of_inc', 'state_location', 'fiscal_year_end', 'filing_html_index', 'htm_filing_link', 'complete_text_filing_link', 'filename', 'item_1', 'item_1A', 'item_1B', 'item_1C', 'item_2', 'item_3', 'item_4', 'item_5', 'item_6', 'item_7', 'item_7A', 'item_8', 'item_9', 'item_9A', 'item_9B', 'item_9C', 'item_10', 'item_11', 'item_12', 'item_13', 'item_14', 'item_15', 'item_16', 'year']
ITEM 2.
PROPERTIES
Our headquarters are located in Mountain View, California. We also own and lease office and building space in the surrounding areas near our headquarters, which in the aggregate (including our headquarters) represent approximately 11.2 million square feet of office/building space and approximately fifty-nine acres of developable land to accommodate anticipated future growth. In addition, we own and lease office/building space and research and development sites around the world


## 3. Score each section with FinBERT

[FinBERT](https://huggingface.co/yiyanghkust/finbert-tone) is a BERT model trained on financial text.
For a piece of text it returns 3 probabilities: **neutral, positive, negative**.

We combine them into one score with a weighted average:

`score = 3 × neutral + 5 × positive + 1 × negative`  → a number between 1 and 5.

*Note: BERT reads at most 512 tokens, so only the start of each section is scored.*

In [5]:
import numpy as np
import pandas as pd
from transformers import BertForSequenceClassification, BertTokenizer

tokenizer = BertTokenizer.from_pretrained("yiyanghkust/finbert-tone")
model = BertForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

def sentiment_score(text):
    inputs = tokenizer(text, max_length=512, truncation=True, return_tensors="pt")
    logits = model(**inputs).logits.detach().numpy()[0]
    probs = np.exp(logits) / np.exp(logits).sum()   # softmax
    return float(np.dot(probs, [3, 5, 1]))

sentiment_score("Revenue grew strongly and margins improved.")

5.000000007069371

In [6]:
SECTIONS = {
    "item_1": "Business Overview", "item_1A": "Risk Factors", "item_1B": "Staff Comments",
    "item_1C": "Cybersecurity", "item_2": "Properties", "item_3": "Legal Proceedings",
    "item_4": "Mine Safety", "item_5": "Equity Market", "item_6": "Financial Summary",
    "item_7": "Management Analysis", "item_7A": "Market Risks", "item_8": "Financial Statements",
    "item_9": "Accountant Changes", "item_9A": "Controls & Procedures", "item_9B": "Other Information",
    "item_9C": "Foreign Inspections", "item_10": "Corporate Governance", "item_11": "Executive Pay",
    "item_12": "Shareholder Info", "item_13": "Related Parties", "item_14": "Audit Fees",
    "item_15": "Exhibits", "item_16": "10-K Summary",
}

rows = []
for filing in filings:
    row = {"Year": filing["year"]}
    for key, name in SECTIONS.items():
        text = filing.get(key, "")
        row[name] = sentiment_score(text) if text.strip() else None
    rows.append(row)

df = pd.DataFrame(rows).dropna(axis=1, how="all")
df["Overall"] = df.drop(columns="Year").mean(axis=1)
df.round(2)

,Year,Business Overview,Risk Factors,Staff Comments,Cybersecurity,Properties,Legal Proceedings,Mine Safety,Equity Market,Financial Summary,...,Other Information,Foreign Inspections,Corporate Governance,Executive Pay,Shareholder Info,Related Parties,Audit Fees,Exhibits,10-K Summary,Overall
0,2018,3.19,1.11,2.98,NaN,3.00,2.86,2.92,3.0,3.00,...,2.98,NaN,3.0,3.0,3.0,3.0,3.0,3.0,2.94,2.89
1,2019,4.99,1.00,2.98,NaN,3.01,2.85,2.92,3.0,3.00,...,2.98,NaN,3.0,3.0,3.0,3.0,3.0,3.0,2.94,2.94
2,2020,4.94,1.00,2.98,NaN,3.01,2.69,2.92,3.0,3.00,...,2.98,NaN,3.0,3.0,3.0,3.0,3.0,3.0,2.94,2.93
3,2021,4.94,1.00,2.98,NaN,3.00,2.72,2.92,3.0,2.79,...,2.97,2.99,3.0,3.0,3.0,3.0,3.0,3.0,2.94,2.93
4,2022,4.99,1.00,2.98,NaN,3.02,2.72,2.85,3.0,2.79,...,2.98,2.99,3.0,3.0,3.0,3.0,3.0,3.0,2.94,2.99
5,2023,5.00,1.00,2.98,3.0,3.04,2.72,2.85,3.0,2.96,...,3.00,3.00,3.0,3.0,3.0,3.0,3.0,3.0,2.99,2.97
6,2024,5.00,1.00,2.98,3.0,3.04,2.72,2.94,3.0,2.97,...,3.00,3.00,3.0,3.0,3.0,3.0,3.0,3.0,2.98,2.96
7,2025,5.00,1.00,2.98,3.0,3.04,2.72,2.94,3.0,2.93,...,3.00,3.00,3.0,3.0,3.0,3.0,3.0,3.0,2.98,2.94


## 4. Visualise the results

In [7]:
import plotly.express as px

fig = px.line(df, x="Year", y="Overall", markers=True, range_y=[1, 5],
              title=f"Overall tone of {company}'s 10-K filings")
fig.add_hline(y=3, line_dash="dot", annotation_text="neutral")
fig.show()

In [8]:
# Heatmap: red = negative, green = positive
score_cols = [c for c in df.columns if c != "Year"]
df.style.background_gradient(cmap="RdYlGn", vmin=1, vmax=5, subset=score_cols).format("{:.2f}", subset=score_cols, na_rep="-")

,Year,Business Overview,Risk Factors,Staff Comments,Cybersecurity,Properties,Legal Proceedings,Mine Safety,Equity Market,Financial Summary,Management Analysis,Market Risks,Financial Statements,Accountant Changes,Controls & Procedures,Other Information,Foreign Inspections,Corporate Governance,Executive Pay,Shareholder Info,Related Parties,Audit Fees,Exhibits,10-K Summary,Overall
0,2018,3.19,1.11,2.98,-,3.00,2.86,2.92,3.00,3.00,3.72,3.00,3.00,1.99,3.03,2.98,-,3.00,3.00,3.00,3.00,3.00,3.00,2.94,2.89
1,2019,4.99,1.00,2.98,-,3.01,2.85,2.92,3.00,3.00,3.03,3.00,3.00,1.99,3.09,2.98,-,3.00,3.00,3.00,3.00,3.00,3.00,2.94,2.94
2,2020,4.94,1.00,2.98,-,3.01,2.69,2.92,3.00,3.00,3.06,3.00,3.00,1.99,3.06,2.98,-,3.00,3.00,3.00,3.00,3.00,3.00,2.94,2.93
3,2021,4.94,1.00,2.98,-,3.00,2.72,2.92,3.00,2.79,3.13,3.00,3.00,1.99,3.10,2.97,2.99,3.00,3.00,3.00,3.00,3.00,3.00,2.94,2.93
4,2022,4.99,1.00,2.98,-,3.02,2.72,2.85,3.00,2.79,4.50,3.00,3.00,1.99,3.04,2.98,2.99,3.00,3.00,3.00,3.00,3.00,3.00,2.94,2.99
5,2023,5.00,1.00,2.98,3.00,3.04,2.72,2.85,3.00,2.96,3.69,3.00,3.00,1.99,3.04,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,2.99,2.97
6,2024,5.00,1.00,2.98,3.00,3.04,2.72,2.94,3.00,2.97,3.43,3.00,3.00,1.99,3.04,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,2.98,2.96
7,2025,5.00,1.00,2.98,3.00,3.04,2.72,2.94,3.00,2.93,3.04,3.00,3.00,1.99,3.04,3.00,3.00,3.00,3.00,3.00,3.00,3.00,3.00,2.98,2.94


In [9]:
# Which sections are the most positive / negative on average?
df[score_cols].drop(columns="Overall").mean().sort_values().round(2)

Risk Factors             1.01
Accountant Changes       1.99
Legal Proceedings        2.75
Mine Safety              2.91
Financial Summary        2.93
10-K Summary             2.96
Staff Comments           2.98
Other Information        2.99
Foreign Inspections      3.00
Market Risks             3.00
Shareholder Info         3.00
Corporate Governance     3.00
Audit Fees               3.00
Executive Pay            3.00
Related Parties          3.00
Exhibits                 3.00
Cybersecurity            3.00
Financial Statements     3.00
Equity Market            3.00
Properties               3.02
Controls & Procedures    3.06
Management Analysis      3.45
Business Overview        4.76
dtype: float64

## 5. Bonus: people mentioned and latest news

* **Named Entity Recognition (NER)** with the [TextRazor](https://www.textrazor.com/) API finds the people named in the filing.
* **[NewsAPI](https://newsapi.org/)** fetches recent news about the company.

Both need a free API key, set as environment variables `TEXTRAZOR_KEY` and `NEWSAPI_KEY`.

In [10]:
import os
import requests

# The Business and Corporate Governance sections name the executives
latest = filings[-1]
text = latest.get("item_1", "") + " " + latest.get("item_10", "")

if not os.getenv("TEXTRAZOR_KEY"):
    print("Set the TEXTRAZOR_KEY environment variable to run this step.")
else:
    response = requests.post(
        "https://api.textrazor.com/",
        headers={"x-textrazor-key": os.getenv("TEXTRAZOR_KEY")},
        data={"text": text[:100_000], "extractors": "entities"},
    )
    entities = response.json().get("response", {}).get("entities", [])
    people = sorted({e["entityId"] for e in entities if "Person" in e.get("type", [])})
    print("People mentioned:", people)

Set the TEXTRAZOR_KEY environment variable to run this step.


In [11]:
if not os.getenv("NEWSAPI_KEY"):
    print("Set the NEWSAPI_KEY environment variable to run this step.")
else:
    response = requests.get(
        "https://newsapi.org/v2/everything",
        params={"q": f'"{company}"', "sortBy": "publishedAt", "pageSize": 5, "apiKey": os.getenv("NEWSAPI_KEY")},
    )
    for article in response.json().get("articles", []):
        print(f"- {article['title']} ({article['source']['name']})\n  {article['url']}")

Set the NEWSAPI_KEY environment variable to run this step.


## Takeaways

* FinBERT turns long, hard-to-read reports into simple numbers we can compare across years.
* For Alphabet (GOOG, 2018-2025), the **Business Overview** is the most positive section (about 4.8) and **Risk Factors** the most negative (about 1.0), as you would expect.
* Most other sections sit close to 3: 10-K reports are written in a careful, neutral tone.
* Limitation: only the first 512 tokens of each section are scored. Splitting sections into chunks and averaging would be more accurate.